# Phase 2: Data Enrichment (Open Food Facts API)

In this phase, we will take our cleaned list of Indian FMCG products and query the Open Food Facts API to find their international counterparts and ingredients.

In [13]:
import pandas as pd
import requests

# loading csv file
df=pd.read_csv("data/multinational_brand_cleaned.csv")
print(f"Dataset loaded successfully!\nTotal products: {len(df)}")

Dataset loaded successfully!
Total products: 172


In [14]:
df.head(10)

,Priority_Rank,S.No,Item name,Brand_Name,Category,Sub_Category,Ingredients,Serving_Size_g,Sugar_g,Total_Fat_g,OFF_UK_Barcode,OFF_UK_Match_Found_Y_N,Indian_Label_Verified_Y_N,Verification_Source_URL,Notes
0,1,184,7 Up Lemon Soft Drink,7 UP,JUICE,SODA,"Carbonated Water, Sugar, Acidity Regulators (3...",100.0,11.7,0,NaN,NaN,NaN,NaN,NaN
1,2,185,7UP Nimbooz Soft Drink,7 UP NIMBOOZ,JUICE,SODA,"Water, Sugar, Concentrated Lemon Juice (0.7%),...",100.0,10.6,0,NaN,NaN,NaN,NaN,NaN
2,3,705,Betty Crocker Triple Chocolate Brownie Instant...,BETTY CROCKER,INSTANT FOOD,CAKE MIX,"Sugar, Refined Wheat Flour (Maida), Cocoa Powd...",100.0,54.3,4.6,NaN,NaN,NaN,NaN,NaN
3,4,593,Bournvita Biscuits,CADBURY,BISCUIT,BISCUIT,"Refined Wheat Flour (Maida) (56%), Sugar, Palm...",100.0,28.3,15.3,NaN,NaN,NaN,NaN,NaN
4,5,391,Bounty Miniatures Coconut Filled Chocolate Pack,BOUNTY,CHOCOLATE,CHOCOLATE BAR,"Milk chocolate coating (44%) (Sugar, Milk soli...",100.0,50,25.8,NaN,NaN,NaN,NaN,NaN
5,6,401,Cadbury Bournville Cranberry 50% Dark CHOCOLAT...,CADBURY,CHOCOLATE,CHOCOLATE BAR,"Sugar, Cocoa Butter (24%), Cocoa Solids (18%),...",100.0,45.7,31.6,NaN,NaN,NaN,NaN,NaN
6,7,402,Cadbury Bournville Fruit & Nut 50% Dark CHOCOL...,CADBURY,CHOCOLATE,CHOCOLATE BAR,"Sugar, Cocoa Butter (24%), Cocoa Solids (18%),...",100.0,44.2,32.8,NaN,NaN,NaN,NaN,NaN
7,8,400,Cadbury Bournville Rich Cocoa 70% Dark CHOCOLA...,CADBURY,CHOCOLATE,CHOCOLATE BAR,"Cocoa Butter, Cocoa Solids, Emulsifiers (442, ...",100.0,20.4,47.4,NaN,NaN,NaN,NaN,NaN
8,9,386,Cadbury Temptations Almond Treat Premium CHOCO...,CADBURY,CHOCOLATE,CHOCOLATE BAR,"Sugar, Almonds (17%), Cocoa Butter, Milk Solid...",100.0,45.2,35,NaN,NaN,NaN,NaN,NaN
9,10,389,Cadbury Temptations Rum & Raisins Premium CHOC...,CADBURY,CHOCOLATE,CHOCOLATE,"Sugar, Raisins (18%), Cocoa Butter, Cocoa Soli...",100.0,57,26,NaN,NaN,NaN,NaN,NaN


In [15]:
df[['Brand_Name', 'Item name']]

,Brand_Name,Item name
0,7 UP,7 Up Lemon Soft Drink
1,7 UP NIMBOOZ,7UP Nimbooz Soft Drink
2,BETTY CROCKER,Betty Crocker Triple Chocolate Brownie Instant...
3,CADBURY,Bournvita Biscuits
4,BOUNTY,Bounty Miniatures Coconut Filled Chocolate Pack
...,...,...
167,TROPICANA,Tropicana Delight Cranberry Fruit Juice
168,TROPICANA,Tropicana Delight Guava Fruit Juice
169,TROPICANA,Tropicana Delight Mixed Fruit Juice
170,TWIX,Twix Minis Cookie Caramel Chocolate Pack


In [46]:
# Test API Request on 1 single product ("7UP")
url = "https://world.openfoodfacts.org/cgi/search.pl"
params = {
    "search_terms" :"7 Up Lemon Soft Drink",
    "search_simple": 1,
    "action": "process",
    "json": 1, #1 means to return the json data not webpage
    "page_size": 10 #just bring 3 results for now
}

# 3. Set User-Agent header (Identifies our project so we don't get blocked as a bot)
headers = {
    "User-Agent": "CrossBorderIngredientPlatform/1.0 (student_research_project)"
}

# 4. Send the GET request over the internet
response = requests.get(url, params=params, headers=headers)
# 5. Check response status code (200 = Success!)
print("HTTP Status Code:", response.status_code)
# 6. Convert response to Python Dictionary
data = response.json() # in this I have multiple items of 7UP

# Print total matches found in Open Food Facts database
#Safe JSON parsing: Using .get('count', 0) is excellent defensive programming. 
# If the API fails and the count key doesn't exist, your code will gracefully return 0 instead of throwing a KeyError and crashing.
print("Total matches found:", data.get('count', 0))

HTTP Status Code: 200
Total matches found: 6


In [47]:
# Inspect the 1st product from our search results
# 1. Get the list of products from our response data dictionary
products=data.get("products",[])
print(f"Number of products returned in this page: {len(products)}\n")

first_product = products[0]

# 3. Extract the key fields we care about
name = first_product.get("product_name", "No Name Found")
barcode = first_product.get("code", "No Barcode")
countries = first_product.get("countries_tags", [])
ingredients = first_product.get("ingredients_text", "No Ingredients Text Found")
# 4. Print the extracted details nicely
print(f"=== 1st Product Found ===")
print(f"Product Name : {name}")
print(f"Barcode      : {barcode}")
print(f"Countries    : {countries}")
print(f"\nIngredients Text:\n{ingredients}")

Number of products returned in this page: 6

=== 1st Product Found ===
Product Name : 7 up
Barcode      : 5410188033809
Countries    : ['en:belgium', 'en:france']

Ingredients Text:
eau gazéifiée, sucre, acidifiants (acide citrique adle malique) aromes naturels de citron et de citron vert, coredelt daddite (citrate de sodium), édulcorant (glycosides de teviol!


In [37]:
# We searched Open Food Facts for "7 Up".
# Open Food Facts did a simple word match and gave us a product named "press up jaouda petit orange" from Morocco! Why? Because "press up" contains the letters "up".
# This is not the 7 Up soda soft drink we wanted!

In [38]:
# If we just naively took products[0] (the very 1st result), we would accidentally pair Indian 7-Up soda with a Moroccan Orange Juice!

# This is why our code needs Smart Filtering Logic:
# When Open Food Facts returns a list of results, we should loop through them and pick the product that:
        # Has a valid ingredients_text (is not empty).
        # Is sold in the UK / Europe / International (e.g. 'en:united-kingdom', 'en:united-states', 'en:france', etc.).

In [48]:
# Cell 4: Loop through all 3 returned products to find the best UK/EU match

print("=== Scanning All Results for a UK/EU Match ===\n")

for i, product in enumerate(products):
    p_name = product.get("product_name", "No Name")
    countries = product.get("countries_tags", [])
    ingredients = product.get("ingredients_text", "")
    
    print(f"Product #{i+1}: {p_name}")
    print(f"  Countries : {countries}")
    print(f"  Has Ingredients? {'Yes' if ingredients else 'No'}")
    print("-" * 50)


=== Scanning All Results for a UK/EU Match ===

Product #1: 7 up
  Countries : ['en:belgium', 'en:france']
  Has Ingredients? Yes
--------------------------------------------------
Product #2: 7 Up tasty twist lemon and lime
  Countries : ['en:united-kingdom']
  Has Ingredients? Yes
--------------------------------------------------
Product #3: Lemon Lime Soda
  Countries : ['en:united-states']
  Has Ingredients? Yes
--------------------------------------------------
Product #4: 7 up free
  Countries : ['en:france', 'en:united-kingdom']
  Has Ingredients? Yes
--------------------------------------------------
Product #5: Seven up
  Countries : ['en:senegal']
  Has Ingredients? Yes
--------------------------------------------------
Product #6: Seven Up
  Countries : ['en:united-arab-emirates']
  Has Ingredients? Yes
--------------------------------------------------


In [61]:
# Cell 5: Version 4 - Waterfall Search Strategy (Think like a Human Analyst!)

NOISE_WORDS = {
    'chocolate', 'bar', 'soft', 'drink', 'flavour', 'flavored', 'flavoured',
    'potato', 'chips', 'pack', 'premium', 'imported', 'instant', 'energy',
    'mini', 'treats', 'candy', 'toffee', 'biscuit', 'wafer', 'delights',
    'crispy', 'original', 'classic', 'special', 'new', 'limited', 'edition'
}

def build_search_queries(brand_name, item_name):
    clean_item = str(item_name).replace('\xa0', ' ').replace('&', 'and').strip()
    
    all_words = clean_item.split()
    core_words = [w for w in all_words if w.lower().rstrip('s') not in NOISE_WORDS]
    
    queries = []
    
    # Query 1: Full item name
    queries.append(clean_item)
    
    # Query 2: First 4 words
    if len(all_words) > 4:
        queries.append(" ".join(all_words[:4]))
        
    # Query 3: First 3 core words
    if core_words and len(core_words) >= 3:
        queries.append(" ".join(core_words[:3]))
        
    # Query 4: First 2 core words (e.g. "Snickers Peanut", "Betty Crocker")
    if core_words and len(core_words) >= 2:
        queries.append(" ".join(core_words[:2]))
        
    # Query 5: Brand + first core word
    core_brand = brand_name.title()
    if core_words:
        queries.append(f"{core_brand} {core_words[0]}")
    
    # Remove duplicates while preserving order
    seen = set()
    unique_queries = []
    for q in queries:
        q_clean = q.strip()
        if q_clean and q_clean not in seen:
            seen.add(q_clean)
            unique_queries.append(q_clean)
            
    return unique_queries


def search_off_api(query, page_size=15):
    """
    ONE single API call for a given query.
    Returns a list of products or empty list.
    """
    url = "https://world.openfoodfacts.org/cgi/search.pl"
    params = {
        "search_terms": query,
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": page_size
    }
    headers = {"User-Agent": "CrossBorderIngredientPlatform/1.0 (student_project)"}
    try:
        response = requests.get(url, params=params, headers=headers, timeout=8)
        if response.status_code == 200:
            return response.json().get("products", [])
    except Exception:
        pass
    return []


def pick_best_product(products):
    """
    From a list of products, pick the best one.
    Priority: UK → US/EU → Any country with ingredients.
    """
    PRIORITY_COUNTRIES = ["united-kingdom", "united-states", "france", "germany", "australia", "canada"]
    
    # Pass 1: Look for UK product
    for p in products:
        countries = str(p.get("countries_tags", [])).lower()
        ingredients = p.get("ingredients_text", "")
        if "united-kingdom" in countries and ingredients and len(ingredients.strip()) > 5:
            return p
    
    # Pass 2: Look for other priority countries
    for country in PRIORITY_COUNTRIES[1:]:
        for p in products:
            countries = str(p.get("countries_tags", [])).lower()
            ingredients = p.get("ingredients_text", "")
            if country in countries and ingredients and len(ingredients.strip()) > 5:
                return p
    
    # Pass 3: Any product with valid ingredients (last resort)
    for p in products:
        ingredients = p.get("ingredients_text", "")
        if ingredients and len(ingredients.strip()) > 5:
            return p
    
    return None


def fetch_uk_ingredients_smart(brand_name, item_name):
    """
    MAIN FUNCTION: Waterfall Search Strategy.
    Tries multiple queries like a human analyst until a match is found.
    """
    queries = build_search_queries(brand_name, item_name)
    
    for query in queries:
        products = search_off_api(query)
        best = pick_best_product(products)
        if best:
            return {
                "OFF_Match_Found": "Yes",
                "OFF_UK_Barcode": best.get("code"),
                "OFF_UK_Ingredients": best.get("ingredients_text", "")
            }
    
    return {"OFF_Match_Found": "No", "OFF_UK_Barcode": None, "OFF_UK_Ingredients": None}


# ---- Quick Test of all 3 problem products ----
print("Test 1 - 7 Up:")
print(fetch_uk_ingredients_smart("7 UP", "7 Up Lemon Soft Drink"))

print("\nTest 2 - Snickers:")
print(fetch_uk_ingredients_smart("MARS", "Snickers Peanut Filled CHOCOLATE BAR"))

print("\nTest 3 - KitKat:")
print(fetch_uk_ingredients_smart("NESTLE", "Nestle KitKat Delights Dark"))

print("\nTest 4 - Coca-Cola Zero:")
print(fetch_uk_ingredients_smart("COCA-COLA", "Coca-Cola Zero Sugar Soft Drink"))


Test 1 - 7 Up:
{'OFF_Match_Found': 'No', 'OFF_UK_Barcode': None, 'OFF_UK_Ingredients': None}

Test 2 - Snickers:
{'OFF_Match_Found': 'Yes', 'OFF_UK_Barcode': '5000159550345', 'OFF_UK_Ingredients': 'Sugar, glucose syrup, peanuts, skimmed milk powder, cocoa butter, cocoa mass, sunflower oil, palm fat, whey permeate (milk), milk fat, salt, emulsifier (soya lecithin) egg white powder.'}

Test 3 - KitKat:
{'OFF_Match_Found': 'Yes', 'OFF_UK_Barcode': '3387390123210', 'OFF_UK_Ingredients': "Farine de BLÉ complet 37,5 %, farine de BLÉ 18,5 %, chocolat en poudre 18,1 % (sucre, cacao en poudre*), semoule de maïs, sirop de glucose, extrait de malt d'ORGE (ORGE, ORGE malté), huile de tournesol, carbonate de calcium, sucre, farine d'ORGE malté, émulsifiant : lécithines ; sel, arômes naturels, fer, vitamines B3, B5, D, B6, B1, B2, B9. Peut contenir du LAIT et des FRUITS À COQUE."}

Test 4 - Coca-Cola Zero:
{'OFF_Match_Found': 'Yes', 'OFF_UK_Barcode': '5449000131805', 'OFF_UK_Ingredients': 'carbonate

In [62]:
# Cell 6: Step 5 & 6 - Batch Process All 172 Products using Waterfall Search & Export

import time

print(f"🚀 Starting Batch API Lookup (Waterfall Strategy) for {len(df)} products...\n")

match_status = []
barcodes = []
uk_ingredients = []

for index, row in df.iterrows():
    brand = row['Brand_Name']
    item = row['Item name']
    
    print(f"[{index + 1}/{len(df)}] Searching: {item[:35]}...", end=" ")
    
    # Call our Waterfall Search function (tries multiple queries like a human)
    result = fetch_uk_ingredients_smart(brand, item)
    
    match_status.append(result['OFF_Match_Found'])
    barcodes.append(result['OFF_UK_Barcode'])
    uk_ingredients.append(result['OFF_UK_Ingredients'])
    
    if result['OFF_Match_Found'] == 'Yes':
        print("✅ MATCH FOUND!")
    else:
        print("❌ Not Found")
        
    # Polite sleep to respect API rate limits
    time.sleep(0.4)

# 1. Update DataFrame columns
df['OFF_UK_Match_Found_Y_N'] = match_status
df['OFF_UK_Barcode'] = barcodes
df['OFF_UK_Ingredients'] = uk_ingredients

# 2. Print Match Summary
total_matches = (df['OFF_UK_Match_Found_Y_N'] == 'Yes').sum()
print("\n" + "="*50)
print(f"🎉 BATCH LOOKUP COMPLETE!")
print(f"Successfully Matched: {total_matches} / {len(df)} products ({(total_matches/len(df))*100:.1f}%)")
print("="*50 + "\n")

# 3. Save Final Files
df.to_csv('data/multinational_brand_enriched.csv', index=False)
df.to_json('data/benchmark_candidates.json', orient='records', indent=4)
print("💾 Enriched datasets saved to data/multinational_brand_enriched.csv and data/benchmark_candidates.json!")

🚀 Starting Batch API Lookup (Waterfall Strategy) for 172 products...

[1/172] Searching: 7 Up Lemon Soft Drink... ✅ MATCH FOUND!
[2/172] Searching: 7UP Nimbooz Soft Drink... ✅ MATCH FOUND!
[3/172] Searching: Betty Crocker Triple Chocolate Brow... ✅ MATCH FOUND!
[4/172] Searching: Bournvita Biscuits... ❌ Not Found
[5/172] Searching: Bounty Miniatures Coconut Filled Ch... ❌ Not Found
[6/172] Searching: Cadbury Bournville Cranberry 50% Da... ✅ MATCH FOUND!
[7/172] Searching: Cadbury Bournville Fruit & Nut 50% ... ✅ MATCH FOUND!
[8/172] Searching: Cadbury Bournville Rich Cocoa 70% D... ✅ MATCH FOUND!
[9/172] Searching: Cadbury Temptations Almond Treat Pr... ✅ MATCH FOUND!
[10/172] Searching: Cadbury Temptations Rum & Raisins P... ✅ MATCH FOUND!
[11/172] Searching: Bournvita Chocolate... ✅ MATCH FOUND!
[12/172] Searching: Cadbury 5 Star 3D CHOCOLATE BAR... ✅ MATCH FOUND!
[13/172] Searching: Cadbury 5 Star CHOCOLATE BAR... ✅ MATCH FOUND!
[14/172] Searching: Cadbury Bournville Rich Cocoa 50% 

In [63]:
# Cell 7: Manual Visual Verification - See Indian vs International Ingredients Side-by-Side

import pandas as pd
pd.set_option('display.max_colwidth', 120)   # Show full ingredient text

matched_df = df[df['OFF_UK_Match_Found_Y_N'] == 'Yes'].reset_index(drop=True)

print(f"✅ Total Products Ready for Visual Inspection: {len(matched_df)}\n")
print("=" * 80)

# Print each product in a clean readable format
for i, row in matched_df.iterrows():
    print(f"\n🔢 Product #{i+1}")
    print(f"   📦 Brand      : {row['Brand_Name']}")
    print(f"   🏷️  Item       : {row['Item name']}")
    print(f"   🇮🇳 Indian ING : {str(row['Ingredients'])[:200]}...")
    print(f"   🌍 Global ING : {str(row['OFF_UK_Ingredients'])[:200]}...")
    print(f"   🔖 Barcode    : {row['OFF_UK_Barcode']}")
    print("-" * 80)
    
    # Pause every 10 products so you can read comfortably
    if (i + 1) % 10 == 0:
        print(f"\n... Showing {i+1}/{len(matched_df)} products so far ...\n")


✅ Total Products Ready for Visual Inspection: 151


🔢 Product #1
   📦 Brand      : 7 UP
   🏷️  Item       : 7 Up Lemon Soft Drink
   🇮🇳 Indian ING : Carbonated Water, Sugar, Acidity Regulators (330,Acidity Regulators 331,Acidity Regulators 296), Preservative (211)...
   🌍 Global ING : carbonated water, sugar, acids (citric acid, malic acid), natural lemon and lime flavouring with other natural flavourings, acidity regulator (sodium citrate), sweetner (steviol glycosides),...
   🔖 Barcode    : 4060800304360
--------------------------------------------------------------------------------

🔢 Product #2
   📦 Brand      : 7 UP NIMBOOZ
   🏷️  Item       : 7UP Nimbooz Soft Drink
   🇮🇳 Indian ING : Water, Sugar, Concentrated Lemon Juice (0.7%), Acidity Regulators 331(Iii),Acidity Regulators 296,Acidity Regulators 330, Flavour (Natural Flavouring Substances), Iodised Salt, Stabilizer (445(Iii)), ...
   🌍 Global ING : WATER, SUGAR, CONCENTRATED LEMON JUICE (0.7%). ACIDITY REGULATORS (331), 296,3